# Ugail–Howard Consciousness Index
## A Mathematical Framework for Quantifying the Dynamics of Consciousness using Hierarchical Integration, Organised Complexity and Metastability

**Authors:** Hassan Ugail & Newton Howard

Cite: H. Ugail and N. Howard, “Quantifying the dynamics of consciousness using
hierarchical integration, organised complexity and metastability,” arXiv preprint
arXiv:2512.10972, Dec. 2025.

---

This notebook is the **standalone core reference implementation** of the Ugail–Howard
Consciousness Index. It provides:

1. the three component estimators — $H_{\mathrm{eff}}$, $D$, $M$;
2. the composite index $\Psi = w_H \cdot H_{\mathrm{eff}} + w_D \cdot D + w_M \cdot M$;
3. a minimal illustrative demonstration on synthetic signals.

It has **no external dataset dependencies** and can be run top-to-bottom in any
Python 3.9+ environment (Google Colab, Jupyter, local venv).

### Composite definition

$$
\Psi \;=\; w_H\,H_{\mathrm{eff}} \;+\; w_D\,D \;+\; w_M\,M,
\qquad (w_H, w_D, w_M) = (0.40,\; 0.35,\; 0.25).
$$

- $H_{\mathrm{eff}}$ — **Scale-free temporal organisation** via the Detrended
  Fluctuation Analysis (DFA) scaling exponent $\alpha_{\mathrm{dfa}}$, mapped
  through a range-normalised triangular tuning:
  $H_{\mathrm{eff}} = \max\!\bigl(0,\; 1 - |\alpha_{\mathrm{dfa}}-H_{\mathrm{opt}}|/\alpha_{\mathrm{range}}\bigr).$
- $D$ — **Cross-frequency organisation** as $I_{\varphi,A}\,(1+\lambda \cdot \mathrm{LZ})$
  with $I_{\varphi,A}$ the Tort (2010) modulation index between $\theta$-phase and
  $\gamma$-amplitude, and $\mathrm{LZ}$ the broadband envelope Lempel–Ziv complexity.
- $M$ — **Metastability** as the temporal standard deviation of the Kuramoto order
  parameter over alpha-band phase.

### Companion notebooks
- `Ugail-Howard-Conciousness-Index_State_Simulator.ipynb` — physiologically motivated simulators for nine canonical conscious states.
- `Ugail-Howard-Conciousness-Index_Validation.ipynb` — full Monte Carlo,
  ablation, sensitivity, and Sleep-EDF empirical validation pipeline.


In [ ]:
# Install dependencies (run once per environment; safe to skip if already installed)
# Uncomment the line below if running in a fresh Colab / notebook kernel.
# !pip install -q numpy scipy matplotlib pandas
print("Dependency cell — uncomment pip line above if needed.")


Dependency cell — uncomment pip line above if needed.


In [ ]:
# ── Imports & global configuration ─────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, hilbert

# Reproducibility
np.random.seed(42)

# Framework configuration
# H_opt is the DFA α value deemed "optimally conscious" — here we fix 0.35 as
# the synthetic-domain prior (mean α_dfa of high-consciousness simulated states).
# For empirical EEG, H_opt should be recalibrated per-subject to Wake α_dfa.
CFG = {
    "fs":      250,     # default sampling rate (Hz)
    "H_opt":   0.35,    # DFA α midpoint for the triangular H_eff tuning
    "sigma_H": 0.12,    # fallback denominator when α_range unavailable
    "w_H":     0.40,    # composite weight on H_eff
    "w_D":     0.35,    # composite weight on D
    "w_M":     0.25,    # composite weight on M
    "lam":     1.0,     # λ in D = I_φA · (1 + λ · LZ)
    "GAMMA_HIGH_SYN":  80,   # upper γ bound for synthetic signals (fs=250 → Nyquist 125)
    "GAMMA_HIGH_REAL": 45,   # upper γ bound for Sleep-EDF (fs=100 → Nyquist 50)
}

# Plotting style
plt.rcParams.update({
    "figure.dpi":      120,
    "font.size":       11,
    "axes.titlesize":  12,
    "axes.labelsize":  11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "legend.frameon":    False,
})

print("Configuration loaded.")
print(f"  Psi = {CFG['w_H']:.2f}*H_eff + {CFG['w_D']:.2f}*D + {CFG['w_M']:.2f}*M   (lam={CFG['lam']:.1f})")
print(f"  H_opt = {CFG['H_opt']:.2f}   (synthetic-domain prior)")


Configuration loaded.
  Psi = 0.40*H_eff + 0.35*D + 0.25*M   (lam=1.0)
  H_opt = 0.35   (synthetic-domain prior)


In [ ]:
# ── Basic signal utilities ─────────────────────────────────────────────────

def bandpass_filter(data, fs, low, high, order=4):
    """Zero-phase Butterworth bandpass filter along the last axis."""
    nyq = fs / 2.0
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    return filtfilt(b, a, data, axis=-1)


def generate_pink_noise(n_channels, n_samples):
    """Approximate 1/f pink noise by cumulative-summed white noise.

    Returns a (n_channels, n_samples) array with zero mean and unit variance
    per channel.
    """
    white = np.random.randn(n_channels, n_samples)
    pink = np.cumsum(white, axis=-1)
    pink -= pink.mean(axis=-1, keepdims=True)
    pink /= pink.std(axis=-1, keepdims=True) + 1e-9
    return pink


print("Signal utilities loaded.")


Signal utilities loaded.


In [ ]:
# ── Detrended Fluctuation Analysis (DFA scaling exponent α) ────────────────
# Note: dfa_hurst() returns the DFA *scaling exponent α*, NOT the Hurst
# exponent H. For cumulative-sum signals:
#   α ∈ (0.5, 1)  → persistent long-range correlations
#   α > 1         → non-stationary fBm-like (expected for deep sleep / seizure)
# The relation to Hurst for fBm is H = α − 1; we report α directly throughout.

def dfa_hurst(x, min_win=16, max_win=None, n_win=10):
    """DFA scaling exponent alpha of a 1-D signal x.

    Parameters
    ----------
    x : array-like, shape (T,)
    min_win, max_win : int
        Integration-window range (samples). If max_win is None, uses T // 4.
    n_win : int
        Number of log-spaced window sizes.

    Returns
    -------
    alpha : float
        DFA scaling exponent. 0.5 ~ white noise, 1.0 ~ pink noise,
        >1 ~ fBm-like persistent non-stationary dynamics.
    """
    x = np.asarray(x)
    N = x.size
    if max_win is None:
        max_win = N // 4

    # Cumulative (integrated) signal, then detrended RMS at each scale
    y = np.cumsum(x - x.mean())
    s_vals = np.unique(
        np.logspace(np.log10(min_win), np.log10(max_win), n_win, dtype=int)
    )
    F = []
    for s in s_vals:
        if s < 4:
            continue
        n_seg = N // s
        if n_seg < 2:
            continue
        rms = []
        for i in range(n_seg):
            seg = y[i * s : (i + 1) * s]
            t = np.arange(s)
            p = np.polyfit(t, seg, 1)
            detrended = seg - np.polyval(p, t)
            rms.append(np.sqrt(np.mean(detrended ** 2)))
        if rms:
            F.append(np.mean(rms))
    F = np.array(F)
    if len(F) < 2:
        return 0.5
    s_use = s_vals[: len(F)]
    coeffs = np.polyfit(np.log(s_use), np.log(F), 1)
    return float(coeffs[0])


# Quick sanity check:
#   - white noise (no temporal structure): alpha ~ 0.5
#   - cumulative-summed white noise (integrated → fBm / Brownian-like): alpha ~ 1.5
_wn = np.random.randn(5000)
_bm = np.cumsum(np.random.randn(5000))
print(f"DFA alpha of white noise          (expect ~0.5): {dfa_hurst(_wn):.3f}")
print(f"DFA alpha of integrated white/fBm (expect ~1.5): {dfa_hurst(_bm):.3f}")


DFA alpha of white noise          (expect ~0.5): 0.513
DFA alpha of integrated white/fBm (expect ~1.5): 1.499


In [ ]:
# ── Lempel–Ziv complexity & Tort (2010) Modulation Index ──────────────────

def lempel_ziv_complexity(binary_seq):
    """Normalised Lempel–Ziv complexity of a 1-D binary sequence.

    Returns c(n) / n where c(n) is the number of distinct phrases in the
    LZ76 parsing.
    """
    s = "".join(str(int(b)) for b in binary_seq)
    i, c, l = 0, 1, 1
    n = len(s)
    while True:
        if i + l > n:
            c += 1
            break
        sub = s[i : i + l]
        if sub in s[:i]:
            l += 1
        else:
            i += l
            c += 1
            l = 1
        if i + l > n:
            break
    return c / n


def mutual_information_phase_amp(phase, amp, n_bins=36):
    """Tort et al. (2010) Modulation Index.

    KL divergence of the amplitude-by-phase distribution from uniform, over
    n_bins phase bins. Range: 0 (no PAC) to ~log(n_bins) ~ 3.58 for 36 bins.

    Amplitude guard
    ---------------
    When the amplitude envelope has negligible power (mean |amp| < 1e-3 after
    z-score normalisation of the source), the KL divergence becomes dominated
    by rounding noise rather than genuine PAC. We return 0.0 in that regime
    to prevent spurious inflation of D (cf. Aru et al., 2015, NeuroImage).

    Parameters
    ----------
    phase : 1-D array of phase values in (-pi, pi).
    amp   : 1-D array of amplitude values, same length as phase.
    n_bins : number of phase bins for the modulation-index histogram.
    """
    phase = np.asarray(phase)
    amp   = np.asarray(amp)

    # Amplitude guard: mean amplitude below this threshold -> return 0
    # (Aru et al. 2015 signal-quality pre-check for PAC estimators.)
    if float(amp.mean()) < 1e-3:
        return 0.0

    bins = np.linspace(-np.pi, np.pi, n_bins + 1)
    amp_profile = np.zeros(n_bins)
    for i in range(n_bins):
        mask = (phase >= bins[i]) & (phase < bins[i + 1])
        amp_profile[i] = amp[mask].mean() if mask.sum() > 0 else 0.0
    total = amp_profile.sum()
    if total < 1e-12:
        return 0.0
    p = amp_profile / total
    q = np.ones(n_bins) / n_bins          # uniform reference
    return float(np.sum(p * np.log(p / q + 1e-12)))   # KL divergence


print("Lempel–Ziv complexity and Tort MI loaded.")


Lempel–Ziv complexity and Tort MI loaded.


In [ ]:
# ── Core metric pipeline: H_eff, D, M, Psi ─────────────────────────────────

def compute_metrics(X, fs=250, Hopt=0.35, sigma_H=0.12, lam=1.0, **kwargs):
    """Compute the three components and composite Psi for a multichannel signal.

    Parameters
    ----------
    X : array, shape (C, T)
        Multichannel signal (C channels x T timepoints).
    fs : float
        Sampling frequency (Hz).
    Hopt : float
        Target DFA alpha for maximum H_eff score.
    sigma_H : float
        Fallback width for triangular tuning when alpha_range is not supplied.
    lam : float
        lambda in D = I_{phi,A} * (1 + lambda * LZ).
    alpha_range : float, optional (kwarg)
        Population-level range of alpha_dfa — supplies the triangular denominator.
        If omitted, defaults to 3*sigma_H (single-signal fallback).
    gamma_high : float, optional (kwarg)
        Upper bound of gamma passband. Defaults to 80 Hz (synthetic); use 45 Hz
        for signals originally sampled <=100 Hz (e.g. Sleep-EDF).

    Returns
    -------
    dict with keys:
        H_raw : DFA scaling exponent alpha_dfa (averaged across channels)
        Heff  : scale-free temporal organisation score (range-normalised triangular)
        D     : cross-frequency organisation = I_{phi,A} * (1 + lambda * LZ)
        M     : metastability = std(R(t)) of alpha-band Kuramoto order parameter
        Psi   : single-call composite 0.40*H_eff + 0.35*D + 0.25*M
    """
    C, T = X.shape

    # ── (1) Scale-free temporal organisation ───────────────────────────
    H_vals = np.array([dfa_hurst(X[ch]) for ch in range(C)])
    H_raw  = float(H_vals.mean())

    # Range-normalised triangular tuning — removes the hard 3*sigma cliff that
    # penalised states at |alpha − H_opt| > 3*sigma. When alpha_range is
    # passed in by the caller (population level),
    # H_eff = max(0, 1 − |alpha − H_opt|/alpha_range).
    _alpha_range = kwargs.get("alpha_range", 3.0 * sigma_H)
    Heff = float(max(0.0, 1.0 - abs(H_raw - Hopt) / _alpha_range))

    # ── (2) Cross-frequency organisation D = I_{phi,A} * (1 + lam * LZ) ─
    theta = bandpass_filter(X, fs, 4, 8)
    gamma_high = kwargs.get("gamma_high", 80)
    gamma = bandpass_filter(X, fs, 30, gamma_high)
    theta_phase = np.angle(hilbert(theta, axis=-1))
    gamma_amp   = np.abs(hilbert(gamma, axis=-1))
    mi_vals = [
        mutual_information_phase_amp(theta_phase[ch], gamma_amp[ch])
        for ch in range(C)
    ]
    I_phiA = float(np.mean(mi_vals))

    # Broadband envelope LZ complexity
    broadband = bandpass_filter(X, fs, 1, 40)
    amp_bb = np.abs(hilbert(broadband, axis=-1))
    LZ_vals = []
    for ch in range(C):
        env = amp_bb[ch]
        binary = (env > np.median(env)).astype(int)
        LZ_vals.append(lempel_ziv_complexity(binary))
    LZ = float(np.mean(LZ_vals))

    D = I_phiA * (1.0 + lam * LZ)

    # ── (3) Metastability M = std(R(t)) ────────────────────────────────
    alpha_band = bandpass_filter(X, fs, 8, 13)
    alpha_phase = np.angle(hilbert(alpha_band, axis=-1))
    R_t = np.abs(np.mean(np.exp(1j * alpha_phase), axis=0))
    M = float(np.std(R_t))

    # ── (4) Single-call composite Psi ──────────────────────────────────
    # Population-level analyses should renormalise H_eff with alpha_range across
    # the full ensemble — this return value is a single-signal convenience.
    Psi = 0.40 * Heff + 0.35 * D + 0.25 * M

    return {"H_raw": H_raw, "Heff": Heff, "D": D, "M": M, "Psi": Psi}


def compute_all_metrics(X, fs, gamma_high=80):
    """Convenience wrapper that returns the raw (non-normalised) components.

    H_eff is deliberately *not* returned here — downstream population-level
    analyses should recompute it with the alpha_range observed in the ensemble.
    """
    m = compute_metrics(X, fs=fs, Hopt=0.35, sigma_H=0.12,
                        lam=1.0, gamma_high=gamma_high)
    return {"H": m["H_raw"], "D": m["D"], "M": m["M"]}


print("Core metric pipeline ready:")
print("  compute_metrics(X, fs, ...)     -> H_raw, H_eff, D, M, Psi")
print("  compute_all_metrics(X, fs, ...) -> H, D, M   (for ensemble analyses)")


Core metric pipeline ready:
  compute_metrics(X, fs, ...)     -> H_raw, H_eff, D, M, Psi
  compute_all_metrics(X, fs, ...) -> H, D, M   (for ensemble analyses)


In [ ]:
# ── Population-level H_eff and normalisation helpers ───────────────────────

def heff_from_population(H_values, H_opt=0.35):
    """Apply the range-normalised triangular H_eff tuning across a population.

    Given an array of DFA alpha values (one per realisation / subject / epoch),
    returns a matching array of H_eff scores using the observed alpha range as
    the triangular denominator:

        alpha_range = max(alpha) − min(alpha)
        H_eff_i = max(0, 1 − |alpha_i − H_opt| / alpha_range)

    This removes the hard cliff imposed by the fixed 3*sigma formula and gives
    a smooth proportional score to every observation.
    """
    H_values = np.asarray(H_values, dtype=float)
    alpha_range = float(H_values.max() - H_values.min())
    if alpha_range < 1e-9:
        return np.zeros_like(H_values)
    return np.maximum(0.0, 1.0 - np.abs(H_values - H_opt) / alpha_range)


def minmax_normalise(values):
    """Min–max scale a 1-D array to [0, 1]. Returns zeros if constant."""
    v = np.asarray(values, dtype=float)
    v_min, v_max = v.min(), v.max()
    return (v - v_min) / (v_max - v_min + 1e-12) if v_max > v_min else np.zeros_like(v)


def psi_from_components(H_eff, D, M, w_H=0.40, w_D=0.35, w_M=0.25):
    """Composite Psi from already-normalised components (each in [0,1])."""
    return w_H * np.asarray(H_eff) + w_D * np.asarray(D) + w_M * np.asarray(M)


print("Population-level helpers loaded.")


Population-level helpers loaded.


In [ ]:
# ── Illustrative demonstration: three reference signals ───────────────────
#
# We exercise the pipeline on three signals chosen to produce distinct Psi
# profiles. This is a sanity check of the index, not a consciousness claim.
#
#   (A) White noise            — no temporal structure, no PAC, no coupling
#   (B) Structured signal      — pink 1/f background + explicit theta–gamma PAC +
#                                network coupling across channels (wake-like)
#   (C) Highly persistent slow — near-1/f^2 drift-dominated signal
#

fs = CFG["fs"]
T  = 7500         # 30 s at 250 Hz — long enough for stable DFA
C  = 16

# ── (A) White noise ───────────────────────────────────────────────────────
X_white = np.random.randn(C, T)

# ── (B) Structured "wake-like" signal ─────────────────────────────────────
# 1/f pink background + alpha (10 Hz) rhythm + theta-phase modulated gamma amp.
np.random.seed(1)
pink = generate_pink_noise(C, T) * 0.5
t_axis = np.arange(T) / fs
theta_wave = np.sin(2 * np.pi * 6.0  * t_axis)   # 6 Hz theta
alpha_wave = np.sin(2 * np.pi * 10.0 * t_axis + np.random.rand(C, 1))
gamma_carrier = np.sin(2 * np.pi * 40.0 * t_axis)
pac_envelope  = (1.0 + 0.8 * theta_wave) / 2.0
gamma_modulated = pac_envelope * gamma_carrier
X_wake = 0.5 * alpha_wave + 0.3 * theta_wave + 0.3 * gamma_modulated + pink
# Mild cross-channel phase coupling
X_wake = X_wake + 0.2 * np.mean(X_wake, axis=0, keepdims=True)
X_wake = (X_wake - X_wake.mean(axis=-1, keepdims=True)) / (X_wake.std(axis=-1, keepdims=True) + 1e-9)

# ── (C) Slow-drift dominated (anaesthesia / deep sleep analogue) ──────────
np.random.seed(2)
slow = np.cumsum(np.random.randn(C, T), axis=-1)
slow = (slow - slow.mean(axis=-1, keepdims=True)) / (slow.std(axis=-1, keepdims=True) + 1e-9)
X_slow = slow + 0.05 * np.random.randn(C, T)
X_slow = (X_slow - X_slow.mean(axis=-1, keepdims=True)) / (X_slow.std(axis=-1, keepdims=True) + 1e-9)

# ── Compute raw components for all three ──────────────────────────────────
rows = []
for name, X in [("White noise", X_white),
                ("Structured (wake-like)", X_wake),
                ("Slow-drift", X_slow)]:
    m = compute_all_metrics(X, fs, gamma_high=CFG["GAMMA_HIGH_SYN"])
    m["Signal"] = name
    rows.append(m)

raw_df = pd.DataFrame(rows)[["Signal", "H", "D", "M"]]
raw_df["H_eff"]   = heff_from_population(raw_df["H"].values, H_opt=CFG["H_opt"])
raw_df["H_eff_n"] = minmax_normalise(raw_df["H_eff"].values)
raw_df["D_n"]     = minmax_normalise(raw_df["D"].values)
raw_df["M_n"]     = minmax_normalise(raw_df["M"].values)
raw_df["Psi"]     = psi_from_components(raw_df["H_eff_n"], raw_df["D_n"], raw_df["M_n"],
                                        w_H=CFG["w_H"], w_D=CFG["w_D"], w_M=CFG["w_M"])

print("Raw & normalised metrics:")
print(raw_df.round(3).to_string(index=False))


Raw & normalised metrics:
                Signal     H     D     M  H_eff  H_eff_n   D_n   M_n   Psi
           White noise 0.495 0.001 0.124  0.850    1.000 0.000 1.000 0.650
Structured (wake-like) 0.775 0.155 0.002  0.559    0.658 1.000 0.000 0.613
            Slow-drift 1.460 0.001 0.117  0.000    0.000 0.001 0.943 0.236


In [ ]:
# ── Visualisation of the demonstration ────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(13.5, 6.2), constrained_layout=True)

# Row 1: example waveforms (first channel, first 3 s)
t_show = np.arange(min(750, T)) / fs
for ax, (name, X) in zip(axes[0],
                         [("White noise",           X_white),
                          ("Structured (wake-like)", X_wake),
                          ("Slow-drift",             X_slow)]):
    ax.plot(t_show, X[0, :len(t_show)], lw=0.8, color="#2166AC")
    ax.set_title(name, fontsize=11)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude (z)")

# Row 2 : component bars side by side, then composite Psi, then state space
comp_ax, psi_ax, scatter_ax = axes[1]
x = np.arange(len(raw_df))
width = 0.26
for i, (col, lbl, col_col) in enumerate([
    ("H_eff_n", r"$H_{\mathrm{eff}}$", "#4393C3"),
    ("D_n",     r"$D$",                "#F4A582"),
    ("M_n",     r"$M$",                "#66C2A5"),
]):
    comp_ax.bar(x + (i - 1) * width, raw_df[col].values, width,
                color=col_col, alpha=0.88, label=lbl, edgecolor="white", lw=0.4)
comp_ax.set_xticks(x)
comp_ax.set_xticklabels(raw_df["Signal"].values, rotation=12, fontsize=8.5)
comp_ax.set_ylim(0, 1.15)
comp_ax.set_ylabel("Normalised component (0–1)")
comp_ax.set_title("Components")
comp_ax.legend(fontsize=8, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.18))

psi_ax.bar(x, raw_df["Psi"].values, color=["#800026", "#4393C3", "#B2182B"], alpha=0.88,
           edgecolor="white", lw=0.4)
for xi, v in zip(x, raw_df["Psi"].values):
    psi_ax.text(xi, v + 0.015, f"{v:.2f}", ha="center", va="bottom", fontsize=9)
psi_ax.set_xticks(x)
psi_ax.set_xticklabels(raw_df["Signal"].values, rotation=12, fontsize=8.5)
psi_ax.set_ylim(0, 1.15)
psi_ax.set_ylabel(r"$\Psi$  (composite)")
psi_ax.set_title(r"Consciousness Index  $\Psi$")

# Scatter: H_eff vs D, marker size proportional to M
sizes = 250 * (0.2 + raw_df["M_n"].values)
colors = ["#800026", "#4393C3", "#B2182B"]
for (_, row), s, c in zip(raw_df.iterrows(), sizes, colors):
    scatter_ax.scatter(row["H_eff_n"], row["D_n"], s=s, alpha=0.85,
                       edgecolors="black", lw=0.5, label=row["Signal"], color=c)
scatter_ax.set_xlabel(r"$H_{\mathrm{eff}}$  (normalised)")
scatter_ax.set_ylabel(r"$D$  (normalised)")
scatter_ax.set_title(r"State space  ($H_{\mathrm{eff}}$,  $D$,  size $\propto M$)")
scatter_ax.set_xlim(-0.05, 1.1)
scatter_ax.set_ylim(-0.05, 1.1)
scatter_ax.legend(fontsize=7.5, loc="upper left", frameon=False)

fig.suptitle("Ugail–Howard Consciousness Index  —  reference demonstration",
             fontsize=12, fontweight="bold")
plt.show()


## Using this library in your own analysis

The two-line pattern below computes $\Psi$ for any multichannel signal `X`
of shape `(C, T)`:

```python
from scipy.signal import butter, filtfilt, hilbert
# ... (re-use the functions defined above) ...

m    = compute_metrics(X, fs=fs, gamma_high=80)
psi  = m["Psi"]   # single-signal composite
```

For population-level analyses (Monte Carlo, across-subject, across-epoch)
use `compute_all_metrics` + `heff_from_population` + `minmax_normalise` +
`psi_from_components` so that $H_{\mathrm{eff}}$ is normalised against the
observed $\alpha$ range of the ensemble, not against the single-signal
fallback.

### Sampling-rate rule of thumb
- Synthetic / modern high-density EEG (fs ≥ 250 Hz): `gamma_high = 80`
- Legacy EEG at fs = 100 Hz (e.g. Sleep-EDF Cassette): `gamma_high = 45`

### Companion notebooks
- `Ugail-Howard-Conciousness-Index_State_Simulator.ipynb` — physiologically motivated simulators for nine canonical conscious states.
- `gail-Howard-Conciousness-Index_Validation.ipynb` — full Monte Carlo,
  ablation, sensitivity, and Sleep-EDF empirical validation pipeline.